<a href="https://colab.research.google.com/github/LeChatNoir578/FinTech-Fraud-Guard/blob/feature%2Fnlp-phishing/nlp_models/vectorizacion_y_modelo_nlp_phishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# FINTECH FRAUD GUARD — MINERÍA DE TEXTO / NLP (DÍA 2 EN COLAB)
# Responsable: Arón (Líder / Especialista NLP)
# Tarea: Vectorización TF-IDF, Términos Clave, Modelo Naïve Bayes, F1 y FPR
# ==============================================================================

import os
import sys
import joblib
import numpy as np
import pandas as pd
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, f1_score

# 1. VERIFICACIÓN Y DESCARGA DE RECURSOS EN NLTK
for resource in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        pass

# 2. BÚSQUEDA Y CARGA EXCLUSIVA DEL DATASET DEL DÍA 1
def cargar_dataset_dia1():
    rutas_posibles = [
        'dataset_texto_limpio_100k.csv',
        'clean_phishing_dataset.csv',
        'data/dataset_texto_limpio_100k.csv',
        'src/ai_models/nlp_phishing/data/dataset_texto_limpio_100k.csv'
    ]

    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            print(f"✅ ¡Dataset del Día 1 cargado desde: '{ruta}'!")
            return pd.read_csv(ruta)

    # SI NO SE ENCUENTRA NINGÚN ARCHIVO, SE EMITE LA ALERTA Y SE DETIENE EL SCRIPT
    print("\n" + "="*80)
    print("[ALERTA DE ERROR]: NO SE ENCONTRÓ EL DATASET DE TEXTO LIMPIO DEL DÍA 1.")
    print("   Por favor, sube tu archivo 'dataset_texto_limpio_100k.csv' o 'clean_phishing_dataset.csv'")
    print("   al panel izquierdo de Google Colab ( Archivos) e intenta ejecutar la celda de nuevo.")
    print("="*80 + "\n")
    sys.exit("Ejecución detenida: Falta el dataset procesado de entrada.")

# Cargar dataset
df = cargar_dataset_dia1()

# 3. VERIFICACIÓN Y LIMPIEZA DE COLUMNAS
col_texto = 'texto_preprocesado' if 'texto_preprocesado' in df.columns else df.columns[-1]
df[col_texto] = df[col_texto].fillna('')

if 'label' not in df.columns:
    print(" [ALERTA DE ERROR]: El dataset cargado no contiene la columna 'label' para clasificar.")
    sys.exit("Ejecución detenida: No se encontró la columna objetivo 'label'.")

print(f"Total de registros a procesar para TF-IDF: {len(df):,}")

# 4. VECTORIZACIÓN TF-IDF (EXTRACCIÓN DE CARACTERÍSTICAS)
print("\n🔠 Generando matriz TF-IDF...")
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df[col_texto])
feature_names = vectorizer.get_feature_names_out()

# Extraer y guardar tabla de términos clave
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
df_terms = pd.DataFrame({
    'termino': feature_names,
    'peso_tfidf_promedio': mean_tfidf
}).sort_values(by='peso_tfidf_promedio', ascending=False)

print("\n--- TOP 10 TÉRMINOS CLAVE EXTRAÍDOS (TF-IDF) ---")
print(df_terms.head(10).to_string(index=False))

# 5. MODELADO INICIAL Y EVALUACIÓN DE MÉTRICAS (F1-SCORE Y FPR)
print("\n🤖 Entrenando clasificador Multinomial Naïve Bayes...")
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df['label'], test_size=0.3, random_state=42, stratify=df['label']
)

model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Matriz de Confusión y Cálculo Explícito de FPR
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# Tasa de Falsos Positivos: FPR = FP / (FP + TN)
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n================ RESULTADOS DEL MODELO ================")
print(f"📊 F1-Score: {f1:.4f}")
print(f"🎯 Tasa de Falsos Positivos (FPR = FP / (FP + TN)): {fpr:.4f}")
print(f"Matriz de Confusión [TN: {tn}, FP: {fp} / FN: {fn}, TP: {tp}]")
print("=======================================================")

# 6. EXPORTACIÓN DE ENTREGABLES
os.makedirs("saved_models", exist_ok=True)
joblib.dump(model, "saved_models/phishing_model_v1.joblib")
joblib.dump(vectorizer, "saved_models/tfidf_vectorizer_v1.joblib")
df_terms.to_csv("vectorizer_terms.csv", index=False)

print("\n💾 Archivos exportados en Colab: 'phishing_model_v1.joblib', 'tfidf_vectorizer_v1.joblib' y 'vectorizer_terms.csv'")

✅ ¡Dataset del Día 1 cargado desde: 'dataset_texto_limpio_100k.csv'!
Total de registros a procesar para TF-IDF: 100,000

🔠 Generando matriz TF-IDF...

--- TOP 10 TÉRMINOS CLAVE EXTRAÍDOS (TF-IDF) ---
termino  peso_tfidf_promedio
    ref             0.110572
   hell             0.083076
     hi             0.079024
regards             0.068970
 thanks             0.061898
 verify             0.058856
   need             0.057974
   team             0.056298
  limit             0.053084
    ple             0.050214

🤖 Entrenando clasificador Multinomial Naïve Bayes...

================ RESULTADOS DEL MODELO ================
📊 F1-Score: 1.0000
🎯 Tasa de Falsos Positivos (FPR = FP / (FP + TN)): 0.0000
Matriz de Confusión [TN: 15000, FP: 0 / FN: 0, TP: 15000]

💾 Archivos exportados en Colab: 'phishing_model_v1.joblib', 'tfidf_vectorizer_v1.joblib' y 'vectorizer_terms.csv'
